# Model Training & Evaluation for Diabetes Risk Prediction

This notebook demonstrates training multiple ML models and evaluating their performance on diabetes risk prediction.

## Contents
1. Data Preparation
2. Train Multiple Models
3. Model Evaluation
4. Feature Importance Analysis
5. Model Comparison
6. Save Best Model

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, auc, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Import trainer
from src.models.train import ModelTrainer

print("✅ Libraries loaded successfully")

## 1. Data Preparation

In [ ]:
# Initialize trainer and load data
trainer = ModelTrainer()
X_train, X_test, y_train, y_test = trainer.load_data()

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"\nClass distribution (Training):")
print(f"  Class 0: {(y_train == 0).sum()}")
print(f"  Class 1: {(y_train == 1).sum()}")
print(f"\nClass distribution (Test):")
print(f"  Class 0: {(y_test == 0).sum()}")
print(f"  Class 1: {(y_test == 1).sum()}")

## 2. Train Multiple Models

In [ ]:
# Scale features for training
X_train_scaled = trainer.scaler.fit_transform(X_train)
X_test_scaled = trainer.scaler.transform(X_test)

# Train models
print("Training models...")
print("\n" + "="*50)

lr_model = trainer.train_logistic_regression(X_train_scaled, y_train)
print("✅ Logistic Regression trained")

rf_model = trainer.train_random_forest(X_train, y_train)
print("✅ Random Forest trained")

try:
    xgb_model = trainer.train_xgboost(X_train, y_train)
    print("✅ XGBoost trained")
except Exception as e:
    print(f"⚠️  XGBoost training skipped: {str(e)}")
    xgb_model = None

## 3. Model Evaluation

In [ ]:
# Evaluate models
models = {
    'logistic_regression': (lr_model, X_test_scaled),
    'random_forest': (rf_model, X_test),
    'xgboost': (xgb_model, X_test) if xgb_model else None
}

results = {}

for model_name, model_data in models.items():
    if model_data is None:
        continue
    
    model, X_test_data = model_data
    
    # Predictions
    y_pred = model.predict(X_test_data)
    y_pred_proba = model.predict_proba(X_test_data)[:, 1]
    
    # Metrics
    accuracy = model.score(X_test_data, y_test)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results[model_name] = {
        'accuracy': accuracy,
        'auc': auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"\n{model_name.upper()}")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  AUC-ROC: {auc:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred))

## 4. Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest
if hasattr(rf_model, 'feature_importances_'):
    feature_names = X_train.columns
    importances = rf_model.feature_importances_
    
    # Create dataframe
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("Random Forest Feature Importance:")
    print(importance_df)
    
    # Plot
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df.head(10), x='Importance', y='Feature', palette='viridis')
    plt.title('Top 10 Most Important Features (Random Forest)')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

## 5. Model Comparison

In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    'Model': results.keys(),
    'Accuracy': [results[m]['accuracy'] for m in results.keys()],
    'AUC-ROC': [results[m]['auc'] for m in results.keys()]
})

print("\nModel Comparison:")
print(comparison_df)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
comparison_df.set_index('Model')['Accuracy'].plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0, 1])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

# AUC comparison
comparison_df.set_index('Model')['AUC-ROC'].plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_title('Model AUC-ROC Comparison')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_ylim([0, 1])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## 6. ROC Curves

In [ ]:
# Plot ROC curves
plt.figure(figsize=(10, 8))

colors = ['blue', 'green', 'red']

for (model_name, result), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, result['y_pred_proba'])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{model_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Save Best Model

In [ ]:
# Determine best model
best_model_name = comparison_df.loc[comparison_df['AUC-ROC'].idxmax(), 'Model']
best_auc = comparison_df['AUC-ROC'].max()

print(f"🏆 Best Model: {best_model_name}")
print(f"📊 Best AUC-ROC: {best_auc:.4f}")

# Select best model
if best_model_name == 'logistic_regression':
    trainer.best_model = lr_model
elif best_model_name == 'random_forest':
    trainer.best_model = rf_model
elif best_model_name == 'xgboost':
    trainer.best_model = xgb_model

trainer.best_model_name = best_model_name
trainer.best_score = best_auc

# Save model
trainer.save_model()
print(f"\n✅ Best model saved successfully!")

## 8. Cross-Validation Analysis

In [ ]:
# Perform cross-validation
print("Cross-Validation Results (5-Fold):")
print("=" * 50)

cv_results = {}

for model_name, model_data in models.items():
    if model_data is None:
        continue
    
    model, X_data = model_data
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_data, y_test, cv=5, scoring='roc_auc')
    cv_results[model_name] = cv_scores
    
    print(f"\n{model_name.upper()}")
    print(f"  Fold Scores: {[f'{s:.4f}' for s in cv_scores]}")
    print(f"  Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

print("\n" + "=" * 50)
print("✅ Model training and evaluation complete!")